In [113]:
cleaned_customer_transaction_data = pd.read_csv(r"C:\Users\user\Downloads\Fraudulent_Transaction_Detection_for_Finlora_Company\Finlora_Dataset\artifacts\Cleaned_Data.csv")
from xgboost import XGBClassifier

!pip install xgboost
!pip install xgboost

from xgboost import XGBClassifier


In [98]:
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(
    cleaned_customer_transaction_data['timestamp'],
    errors='coerce'
)

cleaned_customer_transaction_data['hour'] = cleaned_customer_transaction_data['timestamp'].dt.hour
cleaned_customer_transaction_data['day_of_week'] = cleaned_customer_transaction_data['timestamp'].dt.day_name()
cleaned_customer_transaction_data['is_weekend'] = cleaned_customer_transaction_data['day_of_week'].isin(['Saturday','Sunday']).astype(int)
cleaned_customer_transaction_data['month'] = cleaned_customer_transaction_data['timestamp'].dt.month_name()


In [99]:
cleaned_customer_transaction_data.columns


Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='object')

In [100]:
cleaned_customer_transaction_data.columns


Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='object')

In [101]:
cleaned_customer_transaction_data.columns


Index(['Unnamed: 0', 'transaction_id', 'customer_id', 'timestamp',
       'home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'device_id', 'new_device', 'ip_address', 'ip_country',
       'location_mismatch', 'ip_risk_score', 'kyc_tier', 'account_age_days',
       'device_trust_score', 'chargeback_history_count', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend', 'month'],
      dtype='object')

In [102]:
# Creating threshold‑based features from key risk signals
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(
    cleaned_customer_transaction_data['timestamp']
)

cleaned_customer_transaction_data['late_night_hours'] = (
    (cleaned_customer_transaction_data['hour'] > 3) &
    (cleaned_customer_transaction_data['hour'] < 8)
).astype(int)

cleaned_customer_transaction_data['amount_high'] = (
    cleaned_customer_transaction_data['amount_usd'] > 1000
).astype(int)

cleaned_customer_transaction_data['high_ip_risk'] = (
    cleaned_customer_transaction_data['ip_risk_score'] > 0.8
).astype(int)

cleaned_customer_transaction_data['low_device_trust'] = (
    cleaned_customer_transaction_data['device_trust_score'] < 0.5
).astype(int)

cleaned_customer_transaction_data['new_account'] = (
    cleaned_customer_transaction_data['account_age_days'] < 30
).astype(int)

# Correct velocity feature
cleaned_customer_transaction_data['velocity_spike'] = (
    cleaned_customer_transaction_data['txn_velocity_1h'] > 3
).astype(int)

high_risk_signal_features = cleaned_customer_transaction_data[
    [
        'late_night_hours',
        'amount_high',
        'high_ip_risk',
        'low_device_trust',
        'new_account',
        'velocity_spike'
    ]
]

high_risk_signal_features.head()


,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,velocity_spike
0,0,0,0,0,0,0
1,0,0,0,1,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [103]:
# Checking the features in our dataset
list(cleaned_customer_transaction_data.columns)


['Unnamed: 0',
 'transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'home_country_norm',
 'ip_country_norm',
 'kyc_tier_norm',
 'country_mismatch',
 'hour',
 'day_of_week',
 'is_weekend',
 'month',
 'late_night_hours',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'velocity_spike']

In [104]:
# Feature selection pipeline

# Dropping identifier columns (IDs)
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(
    ['transaction_id', 'customer_id', 'device_id', 'ip_address'],
    axis=1
)

# Dropping irrelevant variables
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(
    ['chargeback_history_count', 'month', 'exchange_rate_src_to_dest', 'Unnamed: 0'],
    axis=1
)


Categorical features

Machine‑learning models cannot understand text categories.
You must identify them so you can encode them properly.

In [105]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(
    include=['object','bool']
).columns

categorical_features


Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'day_of_week'],
      dtype='object')

In [106]:
# Defining numerical features
numerical_features = cleaned_customer_transaction_data.select_dtypes(
    include=['int', 'float']
).columns.drop('is_fraud')

numerical_features


Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'is_weekend',
       'late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust',
       'new_account', 'velocity_spike'],
      dtype='object')

In [107]:

print(f"Categorical: {len(categorical_features)}")
print(f"Numerical: {len(numerical_features)}")
print(f"Dataset: {cleaned_customer_transaction_data.shape}")


Categorical: 13
Numerical: 18
Dataset: (10840, 33)


In [108]:
cleaned_customer_transaction_data.columns



Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'home_country_norm', 'ip_country_norm', 'kyc_tier_norm',
       'country_mismatch', 'hour', 'day_of_week', 'is_weekend',
       'late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust',
       'new_account', 'velocity_spike'],
      dtype='object')

In [109]:
cleaned_customer_transaction_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10840 entries, 0 to 10839
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10840 non-null  datetime64[ns, UTC]
 1   home_country         10840 non-null  object             
 2   source_currency      10840 non-null  object             
 3   dest_currency        10840 non-null  object             
 4   channel              10804 non-null  object             
 5   amount_src           10840 non-null  float64            
 6   amount_usd           10840 non-null  float64            
 7   fee                  10840 non-null  float64            
 8   new_device           10840 non-null  bool               
 9   ip_country           10840 non-null  object             
 10  location_mismatch    10840 non-null  bool               
 11  ip_risk_score        10840 non-null  float64            
 12  kyc_tier          

In [117]:
cleaned_customer_transaction_data.to_csv("../Finlora_Dataset/artifacts/Engineered_Data.csv", index=False)